## 1. Flink Architecture (Body)

- Flink is a distributed framework that handles computation of data streams

- Before we deal with how flink handles the datastreams, it is important to first understand the architecture of flink, and the various components it comprises. This articulation will give us the jargon needed to understand the concepts downstream

### Flink Architecture Components

- Let's suppose we deploy a flink job. What happens then?
    - The entry point of your flink app gets run. Let's call this `def main()`
        - This builds a **JobGraph***, which is just a DAG
        - Each component of the job graph is a transformation of data, called **operators**
        - Sample DAG: `Source -> Map -> Sink` with parallelism: 4
        - This DAG and your flink app is bundled into a single JAR file
    
    - Next, you tell your cluster (e.g. K8s) to start a **JobManager (JM)** and provide it the JAR you just built
        - The JM is just a JVM process started by your cluster
        - JM looks at your JobGraph and concludes: to run this with parallelism 4, I need 4 slots (for simplicity)
    
    - JM looks at how many slots it has available to do this job
        - Since there are 0 **TaskManager (TM)**, it tells the Resource Manager (like K8s) that it needs enough TMs to get 4 slots
        - K8s receives this message, and looks at some config file to tell it how many task slots there are per TM (e.g. `taskmanager.numberOfTaskSlots`)
        - It then spins up $N$ TMs, where $N$ is the number of slots requested / number of task slots per TM
        - Once spun up, TM performs a handshake with JM, and JM now knows which TMs are working for it
    
    - With the TMs and the job blueprint, JMs turn the blueprint into an **execution graph**
        - It sends code for each of step of the dag `Source -> Map -> Sink` to the TMs
        - Flink uses **Operator Chaining** to fuse steps into a single thread, and Slot Sharing to allow those threads to coexist in the same resource slot.
        - TMs set up connections with each other if necessary
            - Why is this needed? Sometimes between steps in the DAG, operators cannot be chained (e.g. there is a keyBy, or a repartition called)
            - Thus, we need to assign a different part of the dag to a new task slot, which may be in a different slot, or even a different TM
            - In such cases, we can incur overhead from copying data from one task slot to another, or even network overhead when transferring data from one TM to another
    
    - Inside each TM, there are multiple task slots
        - A slot has access to a fixed % of the TM's total memory, so one task slot crashing never crashes the others
        - Slots are where "parallelism" kicks in. 
            - Since every operator (`Source`, `Map`, `Sink`) can have their own parallelism, every parallel executor of an operator is called a **subtask**
            - If you have parallelism of 10 for `Source`, you need 10 task slots to handle `Source[1], Source[2]...`
            - In operator chaining, if `Source[1] -> Map[1] -> Sink[1]` is recognised as one logical task, they can live in the same task slot!!

- This means that the idea of "parallelism" is not very dynamic in Flink. Flink does not intelligently "redistribute" your data across TMs, even if TMs are idle. Nor does it increase the number of TMs if you have huge amount of processing load

- Because of this, there are a few "gotcha"s to look out for
    - Suppose you read from a topic with 40 partitions. You want to make sure that your Flink app's parallelism is set to a number that is a divisor of 40
        - For example, if you have parallelism of 8, the 40 partitions divides into 40/8=5 partitions per slot
        - If you have parallelism of 7, you end up with 6/7 slots with 6 partitions, and 1/7 slots 4 partitions, leading to skewed resource usage!

    - Even if your slots are nicely divided according to the partitions of your data, you can **STILL** end up with skew if the data itself is skewed
        - Let's suppose we work at Spotify, and we do a `keyBy` artist to count the number of times the artist was streamed in the last 5 mins
        - `keyBy` forces data with the same key into the same TM
        - So the TM that gets the "Taylor Swift" key end up taking on much more load than the TMs that get other artists!

- Since parallelism isn't dynamic in Flink, there are a few ways to "force" your data to fit the number of partitions you have, EVEN IF you have configured your app's parallelism poorly. You simply pay some cost to copy data around
    - Option 1 `keyBy`: All events with the same key go to the same slot. This is ok if all your keys have around the same cardinality. But if you have some keys that have many more entries than others, you can end up with VERY skewed data, even if you make use of all your task slots
    - Option 2 `rebalance`: All events are uniformly distributed between all your task slots, breaking key grouping. This is a **very** expensive operation, because you are copying data between all your TMs
    - Option 3 `rescale`: Similar to rebalance, but tries to distribute tasks evenly **within** a single downstream dag to minimise network copy

    - What is the difference between `rebalance` and `rescale`?
        - Let's suppose we have the same dag `Source -> Map -> Sink`
        - Suppose `Source` and `Map` are fast, but `Sink` is slow
        - Therefore, we have parallelism 2 for Source and Map, but 4 for Sink
        - So now, we have `Source[1] -> Map[1]` and `Source[2] -> Map[2]`
        - In `rebalance`, every Map talks to every sink, giving us 4*2 copies
            - `Map[1] -> Sink[1]`, `Map[1] -> Sink[2]` ... `Map[2] -> Sink[1]`, `Map[1] -> Sink[2]` ...
        - In `rescale`, we try to say `Source[1] -> Map[1]` will talk to `Sink[1], Sink[2]` and `Source[2] -> Map[2]` will talk to `Sink[3], Sink[4]`, minimising the number of network channels!

### State Management in Flink

- We have covered the basics of the physical architecture of Flink above. Now, let's focus on Flink's main selling point: state management and fault tolerance

- What do we mean by state management?
    - Imagine in python, you write a function that does `def increment(x): return x += 1`
    - This is what you call "stateless". The execution environment does not care what $x$ is, or what it returns. All it cares about it taking some input $x$, and incrementing it by 1

    - This sort of stateless execution doesn't work for some use cases:
        - For example, suppose we operate a payment system, and we want to know the number of payments in the last 5 minutes. 
        - We now need to keep track of "state" somewhere that maintains the count we've seen so far
        - Moreover, suppose our kernel crashes while we are incrementing. We lose the entire chunk of data from where we started accumulation to where the app restarts again

    - In Flink, we achieve statefulness automatically!
        - State is stored locally in the TaskManager, often in an embedded DB called RocksDB, so that processing is fast without network calls to an external DB.

- Managing state also means keeping things in memory, and things in memory are lost when apps crash, or pods restart. Therefore, to have statefulness, there must also be **fault tolerance**

- How does flink handle fault tolerance?
    - Besides the aforementioned use of RocksDB, flink uses **checkpoints** (flink managed) and **savepoints** (user managed)
    - Checkpoint:
        - The JobManager signals the TaskManagers to inject special markers called "Barriers" into the data stream at the sources
        - When the barrier flows through the DAG, each operator "snapshots" its current state and writes to a durable backend (S3, HDFS)
        - Once the barrier is written, this "checkpoint" is complete
    - Savepoint:
        - Same mechanism, but user defines when it happens

- This background explains the following valid strategies used for flink app deployments:
    - Deploy with Checkpoints Invalidated: 
        - "Clean slate" deployment
        - Ignore the data that may be missed between now and the last checkpoint. Just go ahead and process from the latest available data
        - If you are processing a 60min window, you will need to accumulate for 60 mins before you get a value
    
    - Restart from Checkpoint
        - Look for the last automatic backup and resume from there
        - You MIGHT miss a few minutes of data (depending on how long the app was down), but ideally the window won't be empty
    
    - Restart from Provided Savepoint
        - Manual restore
        - Restart from my last defined point. Treat it as a "rollback" to some snapshotted state
    
    - Take Savepoint and Deploy with Checkpoints Invalidated 
        - Gracefully stops the app, and takes a savepoint
        - But deploy as if you don't care about the savepoint
        - This is mostly useful "just in case" you realise you needed the savepoint data
    
    - Take Savepoint and Restart from Checkpoint
        - Take a "snapshot" for future reference, but restart from checkpoint instead
    
    - Take Savepoint and Restart from Provided Savepoint
        - Take a snapshot for reference, but restart from another snapshot

    - (Default) Take Savepoint and Restart from Savepoint
        - Take a snapshot for reference, and restart from that snapshot

### Conclusion

- We've looked at Flink's architecture. Let's move on to discuss Flink's notion of "time", and how it uses the architecture to manage state